# Mejora del modelo segmentado — análisis exhaustivo de variables + manejo de nulos

Fase 2: una vez seleccionadas las variables finales por segmento (en `train_transformers_new_dats.ipynb`),
aquí hacemos un **examen exhaustivo de cada variable** y construimos modelos mejorados por segmento
(**unbounce** y **main**) con un tratamiento de nulos que **preserva la información**.

**Principios:**
- **El nulo es información.** Ej.: `user_studies` — un NaN ("no informado") puede convertir distinto que "nivel 1".
  Se trata como **categoría propia** (`MISSING`), nunca se imputa con un valor real.
- **NO imputar media/mediana** en numéricas: destruye señal. Se deja `NaN` (XGBoost lo usa de forma nativa,
  aprende a qué lado mandar los faltantes) y se añaden **flags `_isna`** donde el nulo discrimine.
- **Variables ordinales** (tipo nivel de estudios) → codificar por **orden**, manteniendo `MISSING` separado.
- Antes de modelar: para CADA variable, comprobar **conversión del NULL vs no-null** y la **forma** de la relación.

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
from google.cloud import bigquery
import matplotlib.pyplot as plt

PROJECT_ID = 'bq-pfu-ga4'; BQ_DATASET = 'BQ_PFU_INCIPY'; BQ_TABLE = 'lead_scoring_train'
TARGET = 'y'

client = bigquery.Client(project=PROJECT_ID)
df = client.query(f'SELECT * FROM `{PROJECT_ID}.{BQ_DATASET}.{BQ_TABLE}`').to_dataframe()

# segmento (si no viene): unbounce -> landing, resto -> main
if 'segmento' not in df.columns:
    df['segmento'] = np.where(df['form_name'].astype(str).str.startswith('unbounce'), 'landing', 'main')

print('df:', df.shape, '| base conversion:', round(df[TARGET].mean()*100, 3), '%')
print('segmento:'); print(df['segmento'].value_counts())

In [ ]:
# Variables finales por segmento (tras analisis de redundancia + CV 5-fold).
# CATEGORICAS con NaN=MISSING, SIN IMPUTAR (el nulo es informativo).
# Quitadas por redundancia (V~1.0) sin perdida en CV:
#   - UNBOUNCE: language_site (duplicada con page_location)
#   - MAIN: page_location (V=1.0 con page_name) y user_country (V=0.91 con user_province)
UNBOUNCE_FEATS = ['product_id', 'page_name', 'user_province', 'user_studies',
                  'utm_campaign']
MAIN_FEATS = ['page_name', 'ga_session_number', 'product_id', 'form_name',
              'user_province', 'user_studies']
ORDINAL_MAPS = {}
print('UNBOUNCE:', len(UNBOUNCE_FEATS), '|', UNBOUNCE_FEATS)
print('MAIN    :', len(MAIN_FEATS), '|', MAIN_FEATS)

## 1. Análisis exhaustivo de variables

Para cada variable: tipo, cardinalidad, % de nulos, y lo más importante:
- **¿El NULL es informativo?** conversión de los nulos vs los no-nulos (con su lift).
- **Forma de la relación** con el target: por deciles (numéricas) o por categoría ordenada (categóricas),
  con la categoría `NaN` **explícita** para verla.

In [ ]:
def analizar_variable(df, col, target='y', topk=15):
    if col not in df.columns:
        print(f'\n[!] {col}: NO esta en el df'); return
    s = df[col]; y = df[target].astype(int); base = y.mean()
    print('=' * 78)
    print(f'{col}  | dtype={s.dtype} | nunique={s.nunique()} | %null={s.isna().mean()*100:.1f}%')
    # ¿El NULL discrimina?
    if s.isna().any():
        cn, co_ = y[s.isna()].mean(), y[s.notna()].mean()
        print(f'  NULL  -> conv={cn*100:.2f}% (n={int(s.isna().sum())}) lift={cn/base:.2f}x   |   '
              f'NO-NULL -> conv={co_*100:.2f}%   [el nulo {"SI" if abs(cn-co_)/base>0.15 else "casi no"} discrimina]')
    # Forma de la relacion
    if pd.api.types.is_numeric_dtype(s) and s.nunique() > 12:
        d = s.describe()
        print(f'  num: min={d["min"]:.2f} p25={d["25%"]:.2f} p50={d["50%"]:.2f} p75={d["75%"]:.2f} max={d["max"]:.2f}')
        try:
            b = pd.qcut(s, 5, duplicates='drop')
            g = y.groupby(b, observed=True).agg(['mean', 'count'])
            g['conv%'] = (g['mean']*100).round(2); g['lift'] = (g['mean']/base).round(2)
            print(g[['conv%', 'count', 'lift']].to_string())
        except Exception as e:
            print('  (no se pudo binnear:', e, ')')
    else:
        g = y.groupby(s.astype('object'), dropna=False).agg(['mean', 'count'])
        g['conv%'] = (g['mean']*100).round(2); g['lift'] = (g['mean']/base).round(2)
        print('  conversion por valor (NaN incluido):')
        print(g.sort_values('mean', ascending=False)[['conv%', 'count', 'lift']].head(topk).to_string())

In [ ]:
# Analisis exhaustivo — UNBOUNCE (sobre el subgrupo landing)
land = df[df['segmento'] == 'landing']
print(f'### UNBOUNCE (n={len(land)}, base={land[TARGET].mean()*100:.2f}%) ###')
for c in UNBOUNCE_FEATS:
    analizar_variable(land, c, TARGET)

In [ ]:
# Analisis exhaustivo — MAIN (sobre el subgrupo main)
main = df[df['segmento'] == 'main']
print(f'### MAIN (n={len(main)}, base={main[TARGET].mean()*100:.2f}%) ###')
for c in MAIN_FEATS:
    analizar_variable(main, c, TARGET)

## 2. Manejo de nulos preservando la información

**Reglas:**
- **Numéricas**: se deja `NaN` (XGBoost lo maneja nativo). NO media/mediana. Se añade flag `col__isna`
  cuando el nulo discrimina (visto en la sección 1).
- **Categóricas**: el `NaN` es una **categoría propia** (`MISSING`), no se rellena.
- **Ordinales** (`ORDINAL_MAPS`): se mapea por orden a entero; los no informados quedan `NaN` (= MISSING),
  nunca se asignan a un nivel real.

In [ ]:
def encode_ordinal(frame, col, mapping):
    # mapea por orden; lo no informado/no mapeado queda NaN (MISSING), no se imputa
    return frame[col].map(mapping).astype('Int64')

def build_X_mejorado(frame, num_feats, cat_feats, isna_cols=None, ordinal_maps=None):
    # NO imputa: deja NaN + flags _isna donde el nulo informa + ordinales por orden con MISSING aparte
    out = pd.DataFrame(index=frame.index)
    ordinal_maps = ordinal_maps or {}
    for c in num_feats:
        if c in ordinal_maps:
            out[c] = encode_ordinal(frame, c, ordinal_maps[c])
        else:
            out[c] = pd.to_numeric(frame[c], errors='coerce')
    for c in cat_feats:
        if c in ordinal_maps:
            out[c] = encode_ordinal(frame, c, ordinal_maps[c])          # ordinal -> numerica ordenada
        else:
            out[c] = frame[c].astype(str).replace('nan', np.nan).astype('category')  # NaN = MISSING
    for c in (isna_cols or []):
        out[f'{c}__isna'] = frame[c].isna().astype('int8')
    return out

## 3. Modelos mejorados por segmento

Para cada segmento, con sus variables finales:
1. Construir X **con** y **sin** flags de nulos (para medir si aportan).
2. Tuning (RandomizedSearch) + **calibración** + evaluación honesta en test (PR-AUC, ROC, lift).
3. Importancia por permutación de la versión final.

> Rellena `ISNA_UNB` / `ISNA_MAIN` con las columnas cuyo nulo discrimina (visto en la sección 1).

In [ ]:
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.isotonic import IsotonicRegression
from sklearn.inspection import permutation_importance
import xgboost as xgb

# columnas (de cada segmento) cuyo NULL es informativo -> se les anade flag _isna  (rellenar tras seccion 1)
ISNA_UNB  = []   # p.ej. ['user_studies', 'utm_campaign']
ISNA_MAIN = []

def split_num_cat(df_seg, feats):
    num = [c for c in feats if pd.api.types.is_numeric_dtype(df_seg[c]) and not c.endswith('_id')]
    cat = [c for c in feats if c not in num]
    return num, cat

def entrenar_segmento(df_seg, feats, isna_cols, label):
    y = df_seg[TARGET].astype(int)
    num, cat = split_num_cat(df_seg, feats)
    res = {}
    for tag, isna in [('sin flags', []), ('con flags _isna', isna_cols)]:
        X = build_X_mejorado(df_seg, num, cat, isna_cols=isna, ordinal_maps=ORDINAL_MAPS)
        Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
        Xtr2, Xva, ytr2, yva = train_test_split(Xtr, ytr, test_size=0.15, stratify=ytr, random_state=42)
        spw = (ytr2 == 0).sum() / (ytr2 == 1).sum()
        m = xgb.XGBClassifier(n_estimators=2000, scale_pos_weight=spw, enable_categorical=True,
                              tree_method='hist', eval_metric='aucpr', early_stopping_rounds=50,
                              n_jobs=-1, random_state=42, max_depth=4, learning_rate=0.05,
                              subsample=0.8, colsample_bytree=0.8, min_child_weight=5)
        m.fit(Xtr2, ytr2, eval_set=[(Xva, yva)], verbose=False)
        p = m.predict_proba(Xte)[:, 1]
        top = pd.DataFrame({'y': yte.values, 'd': pd.qcut(p, 10, labels=False, duplicates='drop')}).groupby('d')['y'].mean().iloc[-1]
        print(f'  [{label}] {tag:16s} PR-AUC={average_precision_score(yte,p):.4f} '
              f'ROC={roc_auc_score(yte,p):.4f} lift_top={top/yte.mean():.2f}x  ({X.shape[1]} cols)')
        res[tag] = (m, X, Xte, yte, p)
    return res

print('=== UNBOUNCE ===')
if UNBOUNCE_FEATS:
    res_unb = entrenar_segmento(df[df.segmento == 'landing'], UNBOUNCE_FEATS, ISNA_UNB, 'UNBOUNCE')
print('\n=== MAIN ===')
if MAIN_FEATS:
    res_main = entrenar_segmento(df[df.segmento == 'main'], MAIN_FEATS, ISNA_MAIN, 'MAIN')

## 4. Conclusiones

_(a rellenar tras ejecutar: ¿aportan los flags de nulos? ¿qué variables mandan en cada segmento? mejora vs el modelo base de la fase 1)_

## 5. Entrenamiento final leak-free (train / eval / test = 70 / 10 / 20)

- **Split 70/10/20 estratificado ANTES de nada** (el `test` no se toca hasta el final).
- **Sin leakage**: el `ColumnTransformer` se ajusta **solo con train** y se aplica a eval/test (nunca se fitea con eval/test).
- Preprocesado: numericas `passthrough` (NaN preservado, XGBoost lo usa); categoricas -> `MISSING` (constante, no imputa valor real) + `OneHotEncoder(min_frequency=20)` (agrupa raras / controla cardinalidad).
- **Tuning**: `RandomizedSearchCV` (CV sobre train). **`eval` -> early stopping** del modelo final. **`test` -> evaluacion unica**.
- Sin calibracion: los scores se usan para **ranking de leads**.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold, train_test_split
from sklearn.metrics import average_precision_score, roc_auc_score
import scipy.stats as st
import xgboost as xgb, numpy as np, pandas as pd

SETS_FINAL = {
    'UNBOUNCE': (df[df.segmento == 'landing'], UNBOUNCE_FEATS),
    'MAIN':     (df[df.segmento == 'main'],    MAIN_FEATS),
}

def split_tipos(frame, feats):
    num = [c for c in feats if pd.api.types.is_numeric_dtype(frame[c]) and not c.endswith('_id')]
    return num, [c for c in feats if c not in num]

def prep_X(frame, num, cat):
    X = pd.DataFrame(index=frame.index)
    for c in num:
        X[c] = pd.to_numeric(frame[c], errors='coerce').astype(float)   # NaN preservado (np.nan)
    for c in cat:
        s = frame[c]
        X[c] = pd.Series(np.where(s.isna(), np.nan, s.astype(str)), index=frame.index, dtype=object)  # str + np.nan
    return X

def make_prep(num, cat):
    catp = Pipeline([('miss', SimpleImputer(strategy='constant', fill_value='MISSING')),  # NaN -> MISSING
                     ('ohe', OneHotEncoder(handle_unknown='ignore', min_frequency=20, sparse_output=False))])
    return ColumnTransformer([('num', 'passthrough', num), ('cat', catp, cat)])

In [ ]:
def entrenar(frame, feats, label, n_iter=30, seed=42):
    num, cat = split_tipos(frame, feats)
    X = prep_X(frame, num, cat); y = frame['y'].astype(int)
    # 70 / 10 / 20  (test fuera desde el inicio)
    X_tr, X_tmp, y_tr, y_tmp = train_test_split(X, y, test_size=0.30, stratify=y, random_state=seed)
    X_ev, X_te, y_ev, y_te   = train_test_split(X_tmp, y_tmp, test_size=2/3, stratify=y_tmp, random_state=seed)

    prep = make_prep(num, cat).fit(X_tr)                 # SOLO train -> sin leakage
    A_tr, A_ev, A_te = prep.transform(X_tr), prep.transform(X_ev), prep.transform(X_te)
    spw = (y_tr == 0).sum() / (y_tr == 1).sum()

    # Tuning de hiperparametros (CV sobre train)
    dist = {'max_depth': st.randint(2, 8), 'learning_rate': st.loguniform(1e-2, 3e-1),
            'subsample': st.uniform(0.6, 0.4), 'colsample_bytree': st.uniform(0.6, 0.4),
            'min_child_weight': st.randint(1, 40), 'gamma': st.uniform(0, 5),
            'reg_lambda': st.loguniform(0.1, 10), 'reg_alpha': st.loguniform(1e-2, 10)}
    base = xgb.XGBClassifier(n_estimators=400, tree_method='hist', eval_metric='aucpr',
                             scale_pos_weight=spw, n_jobs=-1, random_state=seed)
    sr = RandomizedSearchCV(base, dist, n_iter=n_iter, scoring='average_precision',
                            cv=StratifiedKFold(4, shuffle=True, random_state=seed), n_jobs=1, random_state=seed)
    sr.fit(A_tr, y_tr)

    # Modelo final: mejores params + EARLY STOPPING en eval
    final = xgb.XGBClassifier(n_estimators=2000, tree_method='hist', eval_metric='aucpr',
                              early_stopping_rounds=50, scale_pos_weight=spw, n_jobs=-1,
                              random_state=seed, **sr.best_params_)
    final.fit(A_tr, y_tr, eval_set=[(A_ev, y_ev)], verbose=False)

    p_te = final.predict_proba(A_te)[:, 1]
    top = pd.DataFrame({'y': y_te.values, 'd': pd.qcut(p_te, 10, labels=False, duplicates='drop')}).groupby('d')['y'].mean().iloc[-1]
    print(f"=== {label} | train/eval/test = {len(y_tr)}/{len(y_ev)}/{len(y_te)} | cv-best PR-AUC={sr.best_score_:.4f} | best_iter={final.best_iteration} ===")
    print(f"  TEST: ROC={roc_auc_score(y_te, p_te):.4f}  PR-AUC={average_precision_score(y_te, p_te):.4f}  lift_top={top/y_te.mean():.2f}x")
    print(f"  best_params={sr.best_params_}")
    return {'prep': prep, 'model': final, 'features': feats}

modelos = {name: entrenar(frame, feats, name) for name, (frame, feats) in SETS_FINAL.items()}

In [ ]:
# Guardar los dos modelos (preprocesador + modelo + variables) para servir
import joblib, os
os.makedirs('models', exist_ok=True)
for name, m in modelos.items():
    path = f"models/lead_scoring_{name.lower()}.joblib"
    joblib.dump({'preprocessor': m['prep'], 'model': m['model'], 'features': m['features']}, path)
    print('guardado:', path, '|', len(m['features']), 'vars')